In [1]:
"""
Compare various sabre algorithms (or general qiskit transpilers)
"""

#import qiskit
#print(qiskit.__qiskit_version__)
#from qiskit.tools.parallel import CPU_COUNT
# print('CPU Count: ', CPU_COUNT)

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.dagcircuit import DAGCircuit, DAGOpNode, DAGInNode, DAGOutNode
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.transpiler import CouplingMap, PassManager
from qiskit.transpiler.passes import Decompose, ApplyLayout
from qiskit.circuit.library.standard_gates.swap import SwapGate

import numpy as np
from copy import copy, deepcopy
import networkx as nx
import sys


In [67]:
"""Extract QuantumCircuit from Qasm file"""
def extract_circuit(filename, path, node_num):
    # compose the input Quantum Circuit
    q = QuantumRegister(node_num, 'q')
    cir_in = QuantumCircuit(q)
    cir_temp = cir_in.from_qasm_file(path+filename)
    cir_in.compose(cir_temp, inplace=True)
    if node_num < cir_temp.num_qubits:
        raise Exception("Cannot compose circuit!") 
        
    return cir_in

def qct_mapping(cir, coupling_map, qct_mapper):
    #cir = deepcopy(circuit)  # input QuantumCircuit

    layout_pass = qct_mapper(coupling_map)  # initialise the sabrelayout with the given ag coupling map
    pm = PassManager([layout_pass, ApplyLayout()])
    
    # permute the circuit by the mapping generated by pm
    newcir = pm.run(cir) 
    return  newcir, pm.property_set["layout"] 

def qct_route(dag_in: DAGCircuit, coupling_map: CouplingMap, qct_router) -> DAGCircuit:

    if qct_router == StochasticSwap:
        route_pass = qct_router(coupling_map)  
    else: #SabreSwap or SQGMSwap
        route_pass = qct_router(coupling_map, heuristic='lookahead')
        
    dag_out = route_pass.run(dag=dag_in)
    return dag_out #swap non-decomposed

def get_router_name(qct_router):
    if router == SabreSwap:
        return "SabreSwap"      
    if router == StochasticSwap:
        return "StochasticSwap"
    return
    

def qct_more(cir_in, coupling_map: CouplingMap, qct_mapper, qct_router, obj='swap_count', repeat=5):

    best_value = None
    for i in range(repeat):
        permuted_circuit, initial_mapping = qct_mapping(cir_in, coupling_map, qct_mapper)
        #print(f"*test{i}: {initial_mapping}")
        dag_in = circuit_to_dag(permuted_circuit)
        dag_out = qct_route(dag_in, coupling_map, qct_router)
        print(f"*test{i}: swap={dag_out.count_ops()['swap']}")
        if obj == 'swap_count':
            num_swap = dag_out.count_ops()['swap']
            if best_value == None or num_swap < best_value:
                best_value = num_swap  
        else: #obj == 'depth_overhead'
            # decompose SwapGate into CXs
            dag_out = Decompose([SwapGate]).run(dag_out)
            depth_out = dag_out.depth()
            print(f'*test{i}: depth={dag_out.depth()}')
            if best_value == None or depth_out < best_value:
                #print(best_value)
                best_value = depth_out  
        
    if obj == 'swap_count':
        return best_value
    else:
        #return best_value - cir_in.depth()    
        return best_value   

In [79]:
import os, time, csv
from ag import qgrid, q20
#from qiskit.transpiler.passes import SabreLayout as qct_mapper
from qiskit.transpiler.passes import SabreSwap as SabreSwap
from qiskit.transpiler.passes import StochasticSwap as StochasticSwap

from sabre_layout import SabreLayout as qct_mapper

"""AG and Benchmarks"""

AG, AG_name = q20(), 'tokyo'
#AG, AG_name = qgrid(2,3), 'qgrid2x3'
#path = '../bench/6Qbench/'
path = '../bench/qiskit_circuit_benchmark/' 

qct_router = SabreSwap
coupling_map = CouplingMap(couplinglist=AG.edges())
obj = 'swap_count'
#obj = 'depth_overhead'
repeat = 5
start = time.time()    
print(time.asctime())

with open(path + AG_name + '_1001' + ".csv", mode="a") as csv_file:
    fieldnames = ["circ_no", "filename", "input_depth",\
                  'SabreSwap',\
                      "numswap"]

    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()

    count = 0
    for filename in os.listdir(path):
        if not filename.endswith('.qasm'): continue
        if filename != 'phase_oracle_14.qasm': continue

        #if count > 2: break

        '''Extract the circuit from qasm files'''
        cir_in = extract_circuit(filename, path, coupling_map.size())
        #print(type(circuit))
        print(count, filename, cir_in.depth(), cir_in.count_ops()['cx'])
        count += 1

        #depth_in, depth_out = QCT_sabre(filename, path, coupling_map, qct_mapper, qct_routers)
        #print(f"depth_in {depth_in} vs. depth_out {depth_out}")
        best_value = qct_more(cir_in, coupling_map, qct_mapper, qct_router, obj, repeat)
        print(f"best value for {obj}: {best_value} and repeat = {repeat}")

    end = time.time()
    # print('The average cx out/in ratio is', round(sum_cx_out/sum_cx_in, 3))
    # print('The average depth out/in ratio is', round(sum_depth_out/sum_depth_in, 3) )
    print('Used time (s):', round(end-start,2), time.asctime() )



Mon Oct  2 11:48:51 2023
0 phase_oracle_14.qasm 10236 6140
*test0: swap=313
*test1: swap=295
*test2: swap=377
*test3: swap=371
*test4: swap=263
best value for swap_count: 263 and repeat = 5
Used time (s): 14.54 Mon Oct  2 11:49:05 2023
